# 00 — Python and Jupyter Environment Setup

**Learning objective:** verify that this notebook is using the intended Python environment, check core AI/data packages, safely inspect environment-variable readiness, and run local smoke tests without connecting to devices or external services.

All sample values are synthetic. This notebook does not install software automatically.

## 1. Prerequisites

- Python 3.10 or newer is recommended for the course.
- Jupyter or another compatible notebook editor with a Python kernel.
- Basic familiarity with opening a notebook and running cells.
- The core package list is in `requirements.txt`. NumPy, pandas, Matplotlib, and scikit-learn are required for later lessons. PyTorch is optional at this setup stage.

If a package is missing, install it into the same environment selected as the notebook kernel, then restart the kernel. No cell in this notebook runs a package installation.

## 2. Learning objectives

By the end, you will be able to:

1. Inspect the Python version, executable, and kernel support.
2. Check package availability and import core libraries.
3. Confirm the working directory and safely create/read a temporary file.
4. Run tiny NumPy, pandas, plotting, and scikit-learn examples.
5. Check whether an environment variable is configured without displaying its value.
6. Produce a final readiness report and identify any missing requirements.

## 3. Concept explanation: Python environment and Jupyter kernel

A **Python environment** is an interpreter plus the packages available to it. A **Jupyter kernel** is the running process that executes notebook code. VS Code can show a notebook while its selected kernel points to a different Python environment than the one where packages were installed. Therefore, checking imports inside the notebook is more useful than checking a different terminal by itself.

A package can be installed but still unavailable to the current kernel. In that case, verify the selected interpreter, install into that interpreter, and restart the kernel. Do not print secret values when checking configuration; report only whether a value is present.

## 4. Environment setup and imports

This cell imports standard-library tools and checks optional packages before importing them. The `try` blocks are narrow and handle only an unavailable optional dependency; unrelated errors remain visible. Required package imports are direct so a missing required dependency is reported clearly by Python.

In [1]:
from __future__ import annotations

import importlib.metadata as metadata
import importlib.util
import os
import platform
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # Use a non-interactive backend for a portable smoke test.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

try:
    import torch
except ImportError:
    torch = None

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

print(f"Python: {platform.python_version()}")
print(f"Executable: {sys.executable}")
print(f"Platform: {platform.platform()}")
kernel_shell = get_ipython()
kernel_is_running = kernel_shell is not None and kernel_shell.__class__.__name__ == "ZMQInteractiveShell"
print(f"ipykernel importable: {importlib.util.find_spec('ipykernel') is not None}")
print(f"Jupyter IPython kernel running: {kernel_is_running}")
print(f"Working directory: {Path.cwd()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"PyTorch available: {torch is not None}")
if torch is not None:
    print(f"PyTorch: {torch.__version__}")
print(f"python-dotenv available: {load_dotenv is not None}")

Python: 3.13.14
Executable: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
Platform: Windows-11-10.0.26300-SP0
ipykernel importable: True
Jupyter IPython kernel running: True
Working directory: C:\Users\v-yaalam\Downloads\vscode_Dell_17-1
NumPy: 2.3.5
pandas: 3.0.1
Matplotlib: 3.10.8
scikit-learn: 1.8.0
PyTorch available: True
PyTorch: 2.12.0+cpu
python-dotenv available: True


### Explanation and expected output

`sys.executable` identifies the interpreter backing this kernel. `Path.cwd()` is the notebook process's current working directory. The version lines confirm that imports succeeded. PyTorch is reported as optional; the other imported libraries are required for the course's core exercises. Exact version strings vary by installation. If a required import fails, the kernel is likely using an environment without that dependency.

## 5. Beginner example: inspect Python and create a file safely

A working-directory check helps explain why relative file paths sometimes fail. This example creates a temporary folder under the current directory, writes one short synthetic line, reads it back, checks the content, and removes the temporary folder automatically.

In [2]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as temporary_directory:
    test_file = Path(temporary_directory) / "environment_check.txt"
    test_file.write_text("synthetic interface status: up\n", encoding="utf-8")
    saved_text = test_file.read_text(encoding="utf-8").strip()
    assert saved_text == "synthetic interface status: up"

print("Temporary file create/read check: passed")
print(f"Temporary artifacts cleaned up: {not Path(temporary_directory).exists()}")

Temporary file create/read check: passed
Temporary artifacts cleaned up: True


### Step-by-step explanation and expected output

- `TemporaryDirectory` creates a disposable directory under the current working directory, so this also checks write permission there.
- `Path` builds a file path without manually joining path separators.
- `write_text` saves UTF-8 text; `read_text` reads it back.
- `assert` verifies the exact expected result. A failed assertion stops the cell with an error rather than reporting a false success.
- Leaving the `with` block removes the temporary directory.

Expected output is two `passed` / `True` lines. A permission error usually means the kernel's current directory is not writable; choose a permitted working directory and restart the notebook there.

## 6. NumPy smoke test

**NumPy** provides efficient arrays and numerical operations. Its array operations are useful for representing numeric features and doing calculations over many values.

In [3]:
interface_utilization = np.array([0.35, 0.60, 0.85], dtype=float)
average_utilization = float(interface_utilization.mean())
print("Synthetic utilization values:", interface_utilization.tolist())
print("Average utilization:", round(average_utilization, 2))
assert np.isclose(average_utilization, 0.60)

Synthetic utilization values: [0.35, 0.6, 0.85]
Average utilization: 0.6


### Explanation and expected output

`interface_utilization` is a numeric array of three synthetic values. `.mean()` computes their arithmetic average; `float` makes the result a regular Python number for display or serialization. The expected average is `0.6`. If values are accidentally strings, numeric calculations may fail or behave differently; convert and validate data at its boundary.

## 7. pandas smoke test: tabular network data

**pandas** provides labeled tables for loading, cleaning, filtering, and summarizing structured data. A **DataFrame** is a table with named columns and indexed rows.

In [4]:
device_health = pd.DataFrame(
    {
        "device": ["router-a", "switch-b", "router-c"],
        "protocol": ["BGP", "LACP", "OSPF"],
        "state": ["Established", "Up", "Down"],
    }
)
print(device_health.to_string(index=False))
print("Rows:", len(device_health), "| Columns:", len(device_health.columns))
assert list(device_health.columns) == ["device", "protocol", "state"]

  device protocol       state
router-a      BGP Established
switch-b     LACP          Up
router-c     OSPF        Down
Rows: 3 | Columns: 3


### Explanation and expected output

The DataFrame holds synthetic device/protocol/state records. Named columns make it easier to filter—for example, finding rows where `state == "Down"`. The output is a three-row table and its shape. Real operational data must be authorized and sanitized before use.

## 8. Matplotlib smoke test: create a readable chart

**Matplotlib** creates plots for exploring and communicating data. This test saves a small chart in a temporary folder, verifies the image file exists and is non-empty, then cleans it up.

In [5]:
with tempfile.TemporaryDirectory() as temporary_directory:
    chart_path = Path(temporary_directory) / "synthetic_utilization.png"
    figure, axis = plt.subplots(figsize=(5, 3))
    axis.plot([1, 2, 3], interface_utilization, marker="o", label="Utilization")
    axis.set(title="Synthetic Interface Utilization", xlabel="Sample", ylabel="Utilization")
    axis.set_ylim(0, 1)
    axis.legend()
    figure.tight_layout()
    figure.savefig(chart_path)
    plt.close(figure)
    chart_created = chart_path.is_file() and chart_path.stat().st_size > 0

print("Chart file created successfully:", chart_created)
assert chart_created

Chart file created successfully: True


### Explanation and expected output

The chart has a title, axis labels, a bounded utilization scale, and a legend. `savefig` writes the image; `close` releases plotting resources. Expected output: `Chart file created successfully: True`. A headless backend is used so saving works without opening a desktop window.

## 9. scikit-learn smoke test: a tiny synthetic classifier

**scikit-learn** provides consistent tools for classical machine learning, preprocessing, and evaluation. This synthetic demonstration trains a small classifier locally; it downloads no data and contacts no service.

In [6]:
features, labels = make_classification(
    n_samples=80,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    random_state=7,
)
classifier = LogisticRegression(random_state=7)
classifier.fit(features, labels)
predictions = classifier.predict(features)
training_accuracy = accuracy_score(labels, predictions)
print("Synthetic samples:", len(labels))
print("Training accuracy:", round(float(training_accuracy), 2))
assert 0.0 <= training_accuracy <= 1.0

Synthetic samples: 80
Training accuracy: 0.88


### Explanation and expected output

`make_classification` creates reproducible synthetic feature rows (`features`) and binary target labels (`labels`). `fit` learns model parameters; `predict` produces labels; `accuracy_score` measures the fraction matching the supplied labels. The exact accuracy is deterministic for the selected package version and seed, but training accuracy is not an estimate of performance on unseen data. Later lessons will use separate test data and better evaluation practice.

## 10. Optional PyTorch check

**PyTorch** is a deep-learning library for tensor computation and neural networks. It is optional for environment setup; CPU use is sufficient for this small check. A GPU, account, or external model is not required.

In [7]:
if torch is None:
    print("PyTorch check skipped: package is not installed (optional).")
else:
    sample_tensor = torch.tensor([1.0, 2.0, 3.0])
    print("PyTorch tensor:", sample_tensor.tolist())
    print("Tensor mean:", round(float(sample_tensor.mean().item()), 2))

PyTorch tensor:

 [1.0, 2.0, 3.0]
Tensor mean: 2.0


### Explanation and expected output

If PyTorch is installed, this creates a CPU tensor and computes its mean (`2.0`). Otherwise, it clearly reports that the optional check was skipped. If you later need GPU acceleration, validate hardware and install instructions for your platform; do not assume a GPU is present.

## 11. Environment-variable loading without revealing secrets

Environment variables keep credentials out of source code. The optional `python-dotenv` package can load values from a local `.env` file. This cell loads a `.env` only if one exists in the working directory or a parent directory, and prints only whether a placeholder variable is present—never its value. Do not commit `.env`; use `.env.example` as a template.

In [8]:
dotenv_path = None
if load_dotenv is not None:
    from dotenv import find_dotenv

    dotenv_path = find_dotenv(usecwd=True) or None
    if dotenv_path:
        load_dotenv(dotenv_path=dotenv_path, override=False)

api_key_configured = bool(os.getenv("AI_API_KEY"))
print("python-dotenv available:", load_dotenv is not None)
print(".env file found:", dotenv_path is not None)
print("AI_API_KEY configured (value hidden):", api_key_configured)

python-dotenv available: True
.env file found: False
AI_API_KEY configured (value hidden): False


### Explanation and expected output

`find_dotenv` searches for a local `.env`; `load_dotenv(..., override=False)` does not overwrite variables already supplied by the process. Only booleans are displayed. An absent key is normal for this lesson because no external API is used. If a later API lesson needs credentials, set them locally and never paste or print their values.

## 12. Common errors and troubleshooting

| Symptom | Why it happens | How to identify and fix it | How to avoid it |
|---|---|---|---|
| `ModuleNotFoundError` | The selected kernel's environment lacks a package. | Check `sys.executable`; install the dependency into that interpreter/environment and restart the kernel. | Select the intended kernel before running notebooks. |
| Import works in terminal but not notebook | Terminal and notebook use different interpreters. | Compare terminal Python with notebook `sys.executable`; switch kernel or install in the kernel environment. | Verify imports inside the notebook. |
| File not found | Relative paths are resolved from the process working directory. | Print `Path.cwd()` and use a deliberate project-relative path. | Keep data under the project and avoid relying on an unknown launch directory. |
| Permission error during file test | Current directory does not allow writing. | Open the workspace in a writable location and rerun. | Use a dedicated project folder with appropriate permissions. |
| `.env` value appears missing | File not found, variable name mismatch, or kernel not restarted. | Confirm `.env` is beside/above the working directory and inspect only presence, never the value. | Keep `.env` local and use exact variable names. |
| Plot window does not appear | This setup uses a non-interactive backend. | The smoke test saves and verifies a temporary image instead. | Use notebook display settings for later interactive plots. |
| PyTorch check skipped | PyTorch is optional and unavailable in this environment. | Continue; install a platform-compatible build only when a lesson requires it. | Do not install a GPU build without checking platform needs. |

The notebook intentionally does not catch broad exceptions. An unexpected programming error should remain visible so it can be fixed.

## 13. Practice exercises

### Beginner
1. Print the Python major and minor version as a tuple.
2. Make a NumPy array with four synthetic interface utilization values and calculate its maximum.
3. Add a `vendor` column to `device_health` and print only rows whose state is `Down`.

### Intermediate
4. Write a function that accepts a list of utilization values and returns their mean; reject an empty list with a clear `ValueError`.
5. Split synthetic classification data into training and test sets, train logistic regression on training data, and report test accuracy.

### Advanced
6. Create an environment report dictionary containing Python version, required-package availability, and whether the API key is configured. Serialize it to JSON without including any secret value.

### Network automation
7. Given synthetic, read-only interface records, return the names of interfaces whose state is `down`. Do not connect to a device or alter configuration.

## 14. Exercise solutions

Solutions follow. Try the exercises first; then run the cell to compare. All records remain synthetic and offline.

In [9]:
import json
from statistics import mean
from sklearn.model_selection import train_test_split

# 1
python_major_minor = sys.version_info[:2]
print("1:", python_major_minor)

# 2
four_utilization_values = np.array([0.20, 0.45, 0.70, 0.90])
print("2 maximum:", float(four_utilization_values.max()))

# 3
device_health["vendor"] = ["Cisco", "Arista", "Juniper"]
down_devices = device_health.loc[device_health["state"] == "Down"]
print("3 down devices:", down_devices["device"].tolist())

# 4
def average_values(values: list[float]) -> float:
    """Return the mean of a non-empty list of numeric values."""
    if not values:
        raise ValueError("values must not be empty")
    return mean(values)

assert np.isclose(average_values([0.2, 0.4, 0.6]), 0.4)
print("4 average:", average_values([0.2, 0.4, 0.6]))

# 5
features_train, features_test, labels_train, labels_test = train_test_split(
    features, labels, test_size=0.25, random_state=7, stratify=labels
)
test_classifier = LogisticRegression(random_state=7).fit(features_train, labels_train)
test_accuracy = accuracy_score(labels_test, test_classifier.predict(features_test))
print("5 test accuracy:", round(float(test_accuracy), 2))

# 6
environment_report = {
    "python_version": platform.python_version(),
    "numpy_available": importlib.util.find_spec("numpy") is not None,
    "pandas_available": importlib.util.find_spec("pandas") is not None,
    "api_key_configured": bool(os.getenv("AI_API_KEY")),
}
print("6 report JSON:")
print(json.dumps(environment_report, indent=2))

# 7
synthetic_interfaces = [
    {"name": "Ethernet1", "state": "up"},
    {"name": "Ethernet2", "state": "down"},
]
def find_down_interfaces(records: list[dict[str, str]]) -> list[str]:
    """Return names of interfaces marked down in supplied records."""
    return [record["name"] for record in records if record.get("state", "").lower() == "down"]

assert find_down_interfaces(synthetic_interfaces) == ["Ethernet2"]
assert find_down_interfaces([]) == []
assert find_down_interfaces([{"name": "Ethernet3"}]) == []
print("7 down interfaces:", find_down_interfaces(synthetic_interfaces))

# Expected-failure check for exercise 4.
try:
    average_values([])
except ValueError as error:
    print("Empty-input validation:", str(error))

1: (3, 13)
2 maximum: 0.9
3 down devices: ['router-c']
4 average: 0.4
5 test accuracy: 0.85
6 report JSON:
{
  "python_version": "3.13.14",
  "numpy_available": true,
  "pandas_available": true,
  "api_key_configured": false
}
7 down interfaces: ['Ethernet2']
Empty-input validation: values must not be empty


### Solution explanation and expected output

The solutions demonstrate version metadata, array aggregation, DataFrame filtering, input validation, a held-out test split, JSON-safe reporting, and read-only filtering of synthetic interface records. The test accuracy is printed rather than assumed to be a particular value. The empty-list case must print `values must not be empty`. If a field is missing from a network record, production code should validate the record schema and report invalid input rather than silently claiming the interface is healthy.

## 15. Mini-project: local environment readiness report

Build a small report from checks already performed. It records only availability and version metadata—never secret values. The report is printed as JSON and is not written to a permanent file.

In [10]:
required_packages = ["numpy", "pandas", "matplotlib", "sklearn", "ipykernel"]
package_status = {
    package: importlib.util.find_spec(package) is not None
    for package in required_packages
}
readiness_report = {
    "python_version": platform.python_version(),
    "python_supported_for_course": sys.version_info >= (3, 10),
    "working_directory": str(Path.cwd()),
    "required_packages": package_status,
    "jupyter_kernel_running": kernel_is_running,
    "pytorch_optional_available": torch is not None,
    "external_credentials_required_for_this_notebook": False,
}
readiness_report["ready_for_core_lessons"] = (
    readiness_report["python_supported_for_course"]
    and all(package_status.values())
    and kernel_is_running
)
print(json.dumps(readiness_report, indent=2))
assert readiness_report["ready_for_core_lessons"], "Install missing required packages in this kernel environment."

{
  "python_version": "3.13.14",
  "python_supported_for_course": true,
  "working_directory": "C:\\Users\\v-yaalam\\Downloads\\vscode_Dell_17-1",
  "required_packages": {
    "numpy": true,
    "pandas": true,
    "matplotlib": true,
    "sklearn": true,
    "ipykernel": true
  },
  "jupyter_kernel_running": true,
  "pytorch_optional_available": true,
  "external_credentials_required_for_this_notebook": false,
  "ready_for_core_lessons": true
}


### Mini-project explanation and expected output

The report checks Python 3.10+, core package importability, and kernel support. `ready_for_core_lessons` is true only when all required local checks pass. The `working_directory` path is local environment metadata; do not share the report if local paths are sensitive. PyTorch is reported separately because it is optional here.

## 16. Key takeaways

1. Notebook code runs in a kernel, which may differ from the terminal interpreter.
2. Check imports and `sys.executable` in the notebook itself.
3. NumPy, pandas, Matplotlib, and scikit-learn serve distinct numerical, tabular, plotting, and ML needs.
4. Synthetic local data makes setup checks repeatable and safe.
5. PyTorch is optional for this environment check.
6. Environment-variable checks should reveal presence, never secret values.
7. Test and report failures explicitly instead of assuming setup succeeded.

## 17. Restart Kernel and Run All verification checklist

- [ ] Select the intended Python kernel in VS Code/Jupyter.
- [ ] Restart the kernel and run every cell from top to bottom.
- [ ] Confirm Python version and `sys.executable` are the expected environment.
- [ ] Confirm NumPy, pandas, Matplotlib, and scikit-learn import and their smoke tests pass.
- [ ] Confirm the temporary-file test passes and cleans up its artifact.
- [ ] Confirm the plot-file smoke test succeeds.
- [ ] Confirm PyTorch is either tested or clearly reported as optional and unavailable.
- [ ] Confirm no API key or other secret value is printed.
- [ ] Confirm the final readiness report says `true` for `ready_for_core_lessons`.

**Completion summary:** this notebook verifies the local Python/Jupyter foundation and demonstrates small, offline checks for the core AI learning stack.